In [5]:
# %% [code]
"""
Stage 2 — coarse candidate detection, using the before/after/slope GeoTIFFs
you already downloaded and pushed to the Hugging Face dataset repo
(sasudo2/landslides), instead of re-querying Earth Engine.

For each incident:
  1. Pull incident_{id}_before.tif, incident_{id}_after.tif, incident_{id}_slope.tif
     from the HF repo (downloaded on demand, cached locally).
  2. Compute NDVI from the before/after Sentinel-2 bands (B4,B3,B2,B8 order,
     matching your download script).
  3. NDVI difference = NDVI_after - NDVI_before.
  4. Resample the slope raster (30 m) onto the before/after grid (10 m) and
     mask by a minimum slope threshold.
  5. Threshold the masked NDVI-drop layer, extract connected-component blobs.
  6. Filter blobs by area and elongation.
  7. Save each incident's surviving candidate ROIs (buffered bbox + centroid)
     as a JSON, and optionally push that JSON back to the HF repo too.

Requirements:
  pip install huggingface_hub rasterio scikit-image --break-system-packages
"""

import os
import json
import numpy as np
import rasterio
from rasterio.warp import reproject, Resampling
from skimage import measure, morphology
from huggingface_hub import hf_hub_download, HfApi
from kaggle_secrets import UserSecretsClient

# ------------------------------------------------------------------ #
# Config
# ------------------------------------------------------------------ #
REPO_ID = "sasudo2/landslides"
REPO_TYPE = "dataset"

CANDIDATE_DIR = "/kaggle/working/candidates"
os.makedirs(CANDIDATE_DIR, exist_ok=True)

# Band order in your downloaded before/after tifs: ['B4','B3','B2','B8'] -> Red, Green, Blue, NIR
RED_BAND_IDX = 1
NIR_BAND_IDX = 4

NDVI_DROP_THRESHOLD = -0.15      # NDVI decrease considered "significant" vegetation loss
MIN_SLOPE_DEG = 15               # ignore change on near-flat terrain
MIN_BLOB_AREA_M2 = 2_000         # ~0.002 km^2 - discard tiny noise
MAX_BLOB_AREA_M2 = 2_000_000     # ~2 km^2 - discard implausibly large blobs
MAX_ELONGATION = 6.0             # major/minor axis ratio ceiling
CANDIDATE_BUFFER_M = 60          # buffer around each candidate blob's bbox

user_secrets = UserSecretsClient()
hf_token = user_secrets.get_secret("huggingface_token")
api = HfApi(token=hf_token)


# ------------------------------------------------------------------ #
# Fetch this incident's rasters from the Hugging Face repo
# ------------------------------------------------------------------ #
def fetch_incident_rasters(incident_id):
    """Downloads (or reuses cached) before/after/slope tifs for one incident."""
    paths = {}
    for label in ("before", "after", "slope"):
        remote_path = f"incident_{incident_id}/incident_{incident_id}_{label}.tif"
        try:
            local_path = hf_hub_download(
                repo_id=REPO_ID,
                repo_type=REPO_TYPE,
                filename=remote_path,
                token=hf_token,
            )
            paths[label] = local_path
        except Exception as e:
            print(f"   Could not fetch {remote_path}: {e}")
            return None
    return paths


# ------------------------------------------------------------------ #
# NDVI + slope masking, done locally with rasterio/numpy
# ------------------------------------------------------------------ #
def compute_ndvi(band_array):
    red = band_array[RED_BAND_IDX - 1].astype(np.float32)
    nir = band_array[NIR_BAND_IDX - 1].astype(np.float32)
    denom = nir + red
    denom[denom == 0] = np.nan
    return (nir - red) / denom


def build_change_mask(before_path, after_path, slope_path):
    with rasterio.open(before_path) as src_before:
        before_arr = src_before.read()
        transform = src_before.transform
        crs = src_before.crs
        shape = (src_before.height, src_before.width)

    with rasterio.open(after_path) as src_after:
        after_arr = src_after.read(out_shape=(src_after.count, *shape), resampling=Resampling.bilinear)

    ndvi_before = compute_ndvi(before_arr)
    ndvi_after = compute_ndvi(after_arr)
    ndvi_diff = ndvi_after - ndvi_before  # negative = vegetation loss

    # Resample slope (30m) onto the before/after grid (10m)
    with rasterio.open(slope_path) as src_slope:
        slope_resampled = np.empty(shape, dtype=np.float32)
        reproject(
            source=rasterio.band(src_slope, 1),
            destination=slope_resampled,
            src_transform=src_slope.transform,
            src_crs=src_slope.crs,
            dst_transform=transform,
            dst_crs=crs,
            resampling=Resampling.bilinear,
        )

    slope_mask = slope_resampled >= MIN_SLOPE_DEG
    change_mask = np.nan_to_num(ndvi_diff) <= NDVI_DROP_THRESHOLD
    combined = change_mask & slope_mask

    return combined, transform, crs


# ------------------------------------------------------------------ #
# Blob extraction + filtering (unchanged logic, now fed a local array)
# ------------------------------------------------------------------ #
def extract_candidate_blobs(mask_bool, transform, pixel_size_m=10):
    mask_bool = morphology.remove_small_objects(mask_bool, min_size=3)
    mask_bool = morphology.binary_closing(mask_bool, morphology.disk(1))

    labeled = measure.label(mask_bool, connectivity=2)
    candidates = []

    for region in measure.regionprops(labeled):
        area_m2 = region.area * (pixel_size_m ** 2)
        if area_m2 < MIN_BLOB_AREA_M2 or area_m2 > MAX_BLOB_AREA_M2:
            continue

        major = region.major_axis_length or 1
        minor = region.minor_axis_length or 1
        elongation = major / max(minor, 1)
        if elongation > MAX_ELONGATION:
            continue

        min_row, min_col, max_row, max_col = region.bbox
        lon_min, lat_max = transform * (min_col, min_row)
        lon_max, lat_min = transform * (max_col, max_row)

        candidates.append({
            "area_m2": area_m2,
            "elongation": elongation,
            "bbox_lonlat": [lon_min, lat_min, lon_max, lat_max],
        })

    candidates.sort(key=lambda c: c["area_m2"], reverse=True)
    return candidates


def buffer_bbox_deg(bbox, buffer_m, lat_for_scale):
    lon_min, lat_min, lon_max, lat_max = bbox
    m_per_deg_lat = 111_320
    m_per_deg_lon = 111_320 * np.cos(np.radians(lat_for_scale))
    dlat = buffer_m / m_per_deg_lat
    dlon = buffer_m / m_per_deg_lon
    return [lon_min - dlon, lat_min - dlat, lon_max + dlon, lat_max + dlat]


# ------------------------------------------------------------------ #
# Per-incident driver
# ------------------------------------------------------------------ #
def find_candidates_for_incident(incident_id, upload_result=True):
    paths = fetch_incident_rasters(incident_id)
    if paths is None:
        print(f"Skipping incident {incident_id} — missing rasters on the hub.")
        return []

    mask_bool, transform, crs = build_change_mask(paths["before"], paths["after"], paths["slope"])
    blobs = extract_candidate_blobs(mask_bool, transform)

    results = []
    for b in blobs:
        center_lat = (b["bbox_lonlat"][1] + b["bbox_lonlat"][3]) / 2
        buffered = buffer_bbox_deg(b["bbox_lonlat"], CANDIDATE_BUFFER_M, center_lat)
        results.append({
            "incident_id": incident_id,
            "area_m2": b["area_m2"],
            "elongation": b["elongation"],
            "bbox_lonlat": buffered,
        })

    out_path = f"{CANDIDATE_DIR}/incident_{incident_id}_candidates.json"
    with open(out_path, "w") as f:
        json.dump(results, f, indent=2)

    print(f"Incident {incident_id}: {len(results)} candidate ROI(s) -> {out_path}")

    if upload_result:
        api.upload_file(
            path_or_fileobj=out_path,
            path_in_repo=f"candidates/incident_{incident_id}_candidates.json",
            repo_id=REPO_ID,
            repo_type=REPO_TYPE,
        )

    return results


# ------------------------------------------------------------------ #
# Example: run over a list of incident ids
# ------------------------------------------------------------------ #
import pandas as pd

input_csv = "/kaggle/input/datasets/sanjayashrestha123/landslide-reproted/landslides_from_2018_to_2026.csv"
df = pd.read_csv(input_csv)

for inc_id in df["id"].iloc[1000:1005]:
    find_candidates_for_incident(inc_id, upload_result = False)

incident_74296/incident_74296_before.tif:   0%|          | 0.00/7.93M [00:00<?, ?B/s]

incident_74296/incident_74296_after.tif:   0%|          | 0.00/7.84M [00:00<?, ?B/s]

incident_74296/incident_74296_slope.tif:   0%|          | 0.00/347k [00:00<?, ?B/s]

Incident 74296: 2 candidate ROI(s) -> /kaggle/working/candidates/incident_74296_candidates.json
Incident 74297: 2 candidate ROI(s) -> /kaggle/working/candidates/incident_74297_candidates.json
Incident 74298: 2 candidate ROI(s) -> /kaggle/working/candidates/incident_74298_candidates.json


incident_74307/incident_74307_before.tif:   0%|          | 0.00/2.27M [00:00<?, ?B/s]

incident_74307/incident_74307_after.tif:   0%|          | 0.00/2.33M [00:00<?, ?B/s]

incident_74307_slope.tif:   0%|          | 0.00/99.4k [00:00<?, ?B/s]

Incident 74307: 10 candidate ROI(s) -> /kaggle/working/candidates/incident_74307_candidates.json
Incident 74277: 10 candidate ROI(s) -> /kaggle/working/candidates/incident_74277_candidates.json
